# 07 \u2014 Aim 2 RNA Inventory (read-only diagnostic)

**Purpose.** Determine *before any modeling code is written* whether the Aim 2 ALT classifier
contrast (`alt_status` \u2208 `{alt_gold_positive, alt_negative_high_conf}`) is statistically
estimable from the RNA we actually have on disk, or whether ALT status is confounded with
cohort / platform / sample type.

**Read-only.** This notebook does **not** modify the manifest, does **not** train any
classifier, and does **not** load any full expression matrix into memory (index/header
inspection only). It writes a prose report to `data/processed/notebook_07_report.md`.

Conventions, file-backed-RNA verification, and decision rule mirror `notebooks/06_aim1_rna_inventory.ipynb` exactly. The expression-source registry is reused unchanged from `src/aim1_inventory.py`.

**Inputs**

- `data/manifests/master_manifest.csv`
- `data/raw/depmap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv` (`ModelID` column only)
- `data/raw/target_os/TARGET-OS.star_tpm.tsv.gz` (header only)

**Output**

- `data/processed/notebook_07_report.md`

In [1]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ \u2014 tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd

log_versions()
print(f'Run: {RUN_TIMESTAMP}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  pyranges: 0.1.4
  openpyxl: 3.1.5
  requests: 2.33.1
  GEOparse: 2.0.4
Run: 2026-05-17T20:43:05Z


## 1 \u2014 Load manifest and subset to the Aim 2 contrast

Aim 2 arms come from `alt_status` itself: `alt_gold_positive` (positive class) and
`alt_negative_high_conf` (negative class). All other `alt_status` values
(`alt_unknown`, `alt_silver`, `alt_proxy`, ...) are excluded from the primary contrast.

Selection is delegated to `src.aim2_inventory.select_aim2_contrast_rows` so this notebook
cannot drift from the ALT-merge logic in `notebooks/03_alt_label_merge.ipynb`.

In [2]:
from src.aim2_inventory import (
    AIM2_ARMS, ALT_POSITIVE_LABEL, ALT_NEGATIVE_LABEL,
    select_aim2_contrast_rows,
)

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
print(f'Manifest rows: {len(manifest)}')

contrast = select_aim2_contrast_rows(manifest)
print(f'Aim 2 ALT classifier contrast rows: {len(contrast)}')
print(f'  (positive label: {ALT_POSITIVE_LABEL}; negative label: {ALT_NEGATIVE_LABEL})')
print()
print('Arm breakdown:')
print(contrast['arm'].value_counts().to_string())
print()
print('Arm \u00d7 cohort:')
print(pd.crosstab(contrast['cohort'], contrast['arm']).to_string())

Manifest rows: 115
Aim 2 ALT classifier contrast rows: 59
  (positive label: alt_gold_positive; negative label: alt_negative_high_conf)

Arm breakdown:
arm
alt_gold_positive         48
alt_negative_high_conf    11

Arm × cohort:
arm                 alt_gold_positive  alt_negative_high_conf
cohort                                                       
Lawlor_2026_GCCRI                  16                       1
PCGP_2014                          11                       2
Taglialatela_2026                   5                       2
de_Nonneville_2022                 16                       6


## 2 \u2014 DepMap RNA membership check

Verify any DepMap `ModelID` in the Aim 2 contrast against the `ModelID` column of
`OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv`. Reads only that one column, never
the matrix body. Data type for this file is **log2(TPM+1), human protein-coding**.

In [3]:
from src.aim2_inventory import (
    DEPMAP_RNA_PATH, DEPMAP_RNA_DATA_TYPE, depmap_rna_model_ids,
)

print(f'Reading ModelID column from: {DEPMAP_RNA_PATH.name}')
depmap_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
print(f'Unique ModelIDs in DepMap RNA file: {len(depmap_models)}')
print(f'Data type: {DEPMAP_RNA_DATA_TYPE}')
print()

depmap_contrast = contrast[contrast['cohort'] == 'DepMap_26Q1'].copy()
if len(depmap_contrast) == 0:
    print('No DepMap samples in the Aim 2 contrast \u2014 nothing to verify against DepMap RNA file.')
else:
    depmap_contrast['file_backed_rna'] = depmap_contrast['sample_id'].isin(depmap_models)
    print('DepMap contrast samples: file-backed RNA per row')
    print(depmap_contrast[['sample_id', 'arm', 'file_backed_rna']].to_string(index=False))
    missing = depmap_contrast[~depmap_contrast['file_backed_rna']]
    if len(missing):
        print(f"\n!! {len(missing)} DepMap sample_id(s) from the contrast are NOT in the RNA file:")
        for _, r in missing.iterrows():
            print(f"   - {r['sample_id']}  (arm={r['arm']})")
    else:
        print('\nAll DepMap contrast samples are present in the DepMap RNA matrix.')

Reading ModelID column from: OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv
Unique ModelIDs in DepMap RNA file: 1719
Data type: log2(TPM+1) protein-coding

No DepMap samples in the Aim 2 contrast — nothing to verify against DepMap RNA file.


## 3 \u2014 Non-DepMap cohorts: does file-backed expression exist at all?

For every non-DepMap cohort in the Aim 2 contrast, consult the shared
`EXPRESSION_SOURCES` registry in `src/aim1_inventory.py`. A `None` registry path is a
deliberate statement that no expression matrix is currently downloaded for that cohort;
any cohort missing from the registry entirely is reported as **no expression file known**.

If a path is registered we also probe header / index columns to confirm sample IDs map.
Per-sample misses are logged rather than silently dropped.

In [4]:
from src.aim2_inventory import (
    EXPRESSION_SOURCES, TARGET_OS_TPM_PATH, target_os_rna_sample_ids,
)

non_depmap_cohorts = sorted(c for c in contrast['cohort'].unique() if c != 'DepMap_26Q1')
print(f'Non-DepMap cohorts in the contrast: {non_depmap_cohorts}')
print()

for cohort in non_depmap_cohorts:
    src_meta = EXPRESSION_SOURCES.get(cohort)
    sub = contrast[contrast['cohort'] == cohort]
    print(f'=== {cohort} ({len(sub)} contrast samples) ===')
    if src_meta is None:
        print('  No entry in EXPRESSION_SOURCES registry \u2192 no expression file known for this cohort.')
        print('  These samples carry ALT labels only and have NO usable RNA in data/raw/.')
    elif src_meta.path is None:
        print(f'  Registry note: {src_meta.description}')
        print('  No expression matrix on disk \u2192 these samples have NO usable RNA in data/raw/.')
    else:
        print(f'  Registered file: {src_meta.path.name}  (data_type: {src_meta.data_type})')
        print('  Probe NOT implemented for this cohort \u2014 add one to src.aim1_inventory if needed.')
    print()

if TARGET_OS_TPM_PATH.exists():
    target_samples = target_os_rna_sample_ids(TARGET_OS_TPM_PATH)
    print(f'TARGET-OS reference: {TARGET_OS_TPM_PATH.name} has {len(target_samples)} sample columns')
    print('  (TARGET-OS is downloaded but not currently joined to the manifest as a cohort.)')
else:
    target_samples = set()
    print('TARGET-OS TPM file not present on disk.')

Non-DepMap cohorts in the contrast: ['Lawlor_2026_GCCRI', 'PCGP_2014', 'Taglialatela_2026', 'de_Nonneville_2022']

=== Lawlor_2026_GCCRI (17 contrast samples) ===
  Registry note: Lawlor 2026 ships SV/IHC labels only; no expression matrix in data/raw/.
  No expression matrix on disk → these samples have NO usable RNA in data/raw/.

=== PCGP_2014 (13 contrast samples) ===
  Registry note: Chen 2014 PCGP cohort ships TP53 SV / ATRX / telomere-FISH tables only; no expression matrix in data/raw/.
  No expression matrix on disk → these samples have NO usable RNA in data/raw/.

=== Taglialatela_2026 (7 contrast samples) ===
  Registry note: Taglialatela 2026 ships a cell-line list with ALT calls only; no expression matrix in data/raw/.
  No expression matrix on disk → these samples have NO usable RNA in data/raw/.

=== de_Nonneville_2022 (22 contrast samples) ===
  Registry note: de Nonneville 2022 ships a tumor ALT-call table (EV3) only; no expression matrix in data/raw/.
  No expression ma

## 4 \u2014 Tidy per-sample inventory

One row per contrast sample. `file_backed_rna` is `True` only when the actual expression
file contains the sample ID; `manifest_has_rna` is preserved separately so disagreements
between the manifest flag and the on-disk evidence are visible.

In [5]:
from src.aim2_inventory import build_inventory_aim2

inventory = build_inventory_aim2(
    contrast,
    depmap_models=depmap_models,
    target_os_samples=target_samples,
)

with pd.option_context('display.max_rows', None, 'display.max_colwidth', 80,
                       'display.width', 200):
    print(inventory.to_string(index=False))

mismatch = inventory[inventory['manifest_has_rna'] != inventory['file_backed_rna']]
print(f'\nRows where manifest_has_rna != file_backed_rna: {len(mismatch)}')
if len(mismatch):
    print('(These are samples the manifest claims have RNA but no on-disk matrix backs the claim,')
    print(' or vice versa. Each one is reported above so nothing is silently dropped.)')

          sample_id             cohort             alt_status sample_type  manifest_has_rna  file_backed_rna                                                                                          expression_source data_type
deNonneville_2523.0 de_Nonneville_2022      alt_gold_positive       tumor             False            False             de Nonneville 2022 ships a tumor ALT-call table (EV3) only; no expression matrix in data/raw/.   unknown
deNonneville_4435.0 de_Nonneville_2022      alt_gold_positive       tumor             False            False             de Nonneville 2022 ships a tumor ALT-call table (EV3) only; no expression matrix in data/raw/.   unknown
deNonneville_4680.0 de_Nonneville_2022      alt_gold_positive       tumor             False            False             de Nonneville 2022 ships a tumor ALT-call table (EV3) only; no expression matrix in data/raw/.   unknown
deNonneville_7695.0 de_Nonneville_2022      alt_gold_positive       tumor             False     

## 5 \u2014 Cross-tabulations restricted to file-backed RNA

In [6]:
fb = inventory[inventory['file_backed_rna']].copy()
print(f'File-backed RNA samples: {len(fb)}  '
      f"(positive={int((fb['alt_status']==ALT_POSITIVE_LABEL).sum())}, "
      f"negative={int((fb['alt_status']==ALT_NEGATIVE_LABEL).sum())})")
print()

def _xtab(df, idx_col, val_col='alt_status'):
    if len(df) == 0:
        return pd.DataFrame()
    return pd.crosstab(df[idx_col], df[val_col])

for idx_col in ('cohort', 'sample_type', 'data_type'):
    print(f'--- {idx_col} \u00d7 alt_status (file_backed_rna only) ---')
    ct = _xtab(fb, idx_col)
    print(ct.to_string() if not ct.empty else '(no rows)')
    print()

File-backed RNA samples: 0  (positive=0, negative=0)

--- cohort × alt_status (file_backed_rna only) ---
(no rows)

--- sample_type × alt_status (file_backed_rna only) ---
(no rows)

--- data_type × alt_status (file_backed_rna only) ---
(no rows)



## 6 \u2014 Confounding assessment

For each arm we report the percentage breakdown by cohort, sample_type, and data_type;
and we evaluate two boolean conditions used by the decision rule:

1. Does at least one cohort contain **both arms** with file-backed RNA?
2. Does at least one cohort contain both arms with file-backed RNA **and the same `data_type`**?

In [7]:
from src.aim2_inventory import assess_confounding_aim2

assessment = assess_confounding_aim2(inventory)

print(f"File-backed RNA totals  \u2192  positive={assessment['n_positive']}, "
      f"negative={assessment['n_negative']}, total={assessment['n_total']}")
print()

def _pct_by(arm_label, col):
    sub = fb[fb['alt_status'] == arm_label]
    if len(sub) == 0:
        return f'   (no file-backed {arm_label} samples)'
    counts = sub[col].value_counts()
    total = counts.sum()
    return '\n'.join(
        f'   {label:<28} {n:>3}  ({100*n/total:5.1f}%)'
        for label, n in counts.items()
    )

for arm in AIM2_ARMS:
    print(f'=== arm = {arm} ===')
    for col in ('cohort', 'sample_type', 'data_type'):
        print(f' by {col}:')
        print(_pct_by(arm, col))
    print()

print('--- co-occurrence flags ---')
print(f"cohorts_with_both_arms                 : {assessment['cohorts_with_both_arms']}")
print(f"cohorts_with_both_arms_same_data_type  : {assessment['cohorts_with_both_arms_same_dtype']}")
print(f"has_within_cohort_co_occurrence              : {assessment['has_within_cohort_co_occurrence']}")
print(f"has_within_cohort_co_occurrence_same_dtype   : {assessment['has_within_cohort_co_occurrence_same_dtype']}")

File-backed RNA totals  →  positive=0, negative=0, total=0

=== arm = alt_gold_positive ===
 by cohort:
   (no file-backed alt_gold_positive samples)
 by sample_type:
   (no file-backed alt_gold_positive samples)
 by data_type:
   (no file-backed alt_gold_positive samples)

=== arm = alt_negative_high_conf ===
 by cohort:
   (no file-backed alt_negative_high_conf samples)
 by sample_type:
   (no file-backed alt_negative_high_conf samples)
 by data_type:
   (no file-backed alt_negative_high_conf samples)

--- co-occurrence flags ---
cohorts_with_both_arms                 : []
cohorts_with_both_arms_same_data_type  : []
has_within_cohort_co_occurrence              : False
has_within_cohort_co_occurrence_same_dtype   : False


## 7 \u2014 Apply the design decision rule

Same three categories as Aim 1, applied in this order:

1. **Within-cohort viable** \u2014 \u22651 (cohort \u00d7 data_type) pair has both arms with \u22653 samples per arm.
2. **Pooled-exploratory only** \u2014 no single cohort has both arms, but each arm spans \u22652 cohorts.
3. **Not estimable as defined** \u2014 anything else (an arm has zero file-backed RNA, or positive
   and negative come from disjoint single cohorts).

In [8]:
from src.aim2_inventory import decide_aim2_design

verdict = decide_aim2_design(inventory, assessment)
print(f'VERDICT: {verdict["category"]}')
print()
print(verdict['justification'])

VERDICT: not_estimable

File-backed RNA is available for 0 ALT-positive sample(s) (from cohort(s) (none)) and 0 ALT-negative sample(s) (from cohort(s) (none)). ALT status is perfectly or near-perfectly confounded with cohort / platform / sample_type, so the Aim 2 contrast as currently defined is not statistically estimable. The contrast must be redefined (e.g. require RNA-backed cohorts on both sides) or restricted (e.g. drop arms without file-backed RNA) before any ALT classifier can be trained.


## 8 \u2014 Write `data/processed/notebook_07_report.md`

In [9]:
REPORT_PATH = ROOT / 'data' / 'processed' / 'notebook_07_report.md'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

VERDICT_LABEL = {
    'within_cohort_viable':     'WITHIN-COHORT VIABLE',
    'pooled_exploratory_only':  'POOLED-EXPLORATORY ONLY',
    'not_estimable':            'NOT ESTIMABLE AS DEFINED',
}[verdict['category']]

def _md_table(df, index_label):
    if df.empty:
        return '(no rows)\n'
    cols = list(df.columns)
    header = '| ' + index_label + ' | ' + ' | '.join(str(c) for c in cols) + ' |\n'
    sep    = '|' + '|'.join(['---'] * (len(cols) + 1)) + '|\n'
    body   = ''.join(
        '| ' + str(idx) + ' | ' + ' | '.join(str(v) for v in row) + ' |\n'
        for idx, row in df.iterrows()
    )
    return header + sep + body

lines: list[str] = []
lines.append('# Notebook 07 \u2014 Aim 2 RNA Inventory\n')
lines.append(f'_Generated: {RUN_TIMESTAMP}_\n')
lines.append('')
lines.append('Read-only diagnostic. Determines whether the Aim 2 ALT classifier contrast '
             '(`alt_status` \u2208 `{alt_gold_positive, alt_negative_high_conf}`) is statistically '
             'estimable from the RNA available on disk, or whether ALT status is confounded '
             'with cohort / platform / sample type.\n')
lines.append('')
lines.append('## Inputs\n')
lines.append(f'- Manifest: `{MANIFEST_PATH.relative_to(ROOT)}` ({len(manifest)} rows)')
lines.append(f'- DepMap RNA: `{DEPMAP_RNA_PATH.relative_to(ROOT)}` ({len(depmap_models)} unique ModelIDs)')
lines.append(f'- TARGET-OS TPM: `{TARGET_OS_TPM_PATH.relative_to(ROOT)}` '
             f'({len(target_samples)} sample columns; not joined to manifest as a cohort)')
lines.append('')
lines.append('## Primary contrast composition\n')
lines.append(f'Total contrast rows: **{len(contrast)}** '
             f"(positive={int((contrast['arm']==ALT_POSITIVE_LABEL).sum())}, "
             f"negative={int((contrast['arm']==ALT_NEGATIVE_LABEL).sum())}).\n")
lines.append('Arm \u00d7 cohort:\n')
lines.append('```')
lines.append(pd.crosstab(contrast['cohort'], contrast['arm']).to_string())
lines.append('```\n')

lines.append('## Per-sample RNA-backing inventory\n')
lines.append('| sample_id | cohort | alt_status | sample_type | manifest_has_rna | file_backed_rna | expression_source | data_type |')
lines.append('|---|---|---|---|---|---|---|---|')
for _, r in inventory.iterrows():
    lines.append('| ' + ' | '.join(str(r[c]) for c in [
        'sample_id', 'cohort', 'alt_status', 'sample_type',
        'manifest_has_rna', 'file_backed_rna', 'expression_source', 'data_type',
    ]) + ' |')
lines.append('')

lines.append('## Cross-tabulations (file-backed RNA only)\n')
for idx_col in ('cohort', 'sample_type', 'data_type'):
    lines.append(f'### `{idx_col}` \u00d7 alt_status\n')
    ct = pd.crosstab(fb[idx_col], fb['alt_status']) if len(fb) else pd.DataFrame()
    lines.append(_md_table(ct, idx_col))

lines.append('## Confounding assessment (file-backed RNA only)\n')
lines.append(f"- File-backed totals: **positive={assessment['n_positive']}**, "
             f"**negative={assessment['n_negative']}**, total={assessment['n_total']}.")
for arm in AIM2_ARMS:
    sub = fb[fb['alt_status'] == arm]
    lines.append(f"- arm `{arm}` ({len(sub)} sample(s)):")
    if len(sub) == 0:
        lines.append('    - (no file-backed samples)')
        continue
    for col in ('cohort', 'sample_type', 'data_type'):
        counts = sub[col].value_counts()
        total = counts.sum()
        bits = '; '.join(f'{lab}: {n} ({100*n/total:.0f}%)' for lab, n in counts.items())
        lines.append(f'    - by {col}: {bits}')
lines.append('')
lines.append(f"- Cohorts with both arms file-backed: `{assessment['cohorts_with_both_arms']}`")
lines.append(f"- Cohorts with both arms file-backed under the same data_type: "
             f"`{assessment['cohorts_with_both_arms_same_dtype']}`")
lines.append(f"- `has_within_cohort_co_occurrence`: **{assessment['has_within_cohort_co_occurrence']}**")
lines.append(f"- `has_within_cohort_co_occurrence_same_dtype`: "
             f"**{assessment['has_within_cohort_co_occurrence_same_dtype']}**")
lines.append('')

lines.append('## AIM 2 DESIGN VERDICT\n')
lines.append(f'**Category: {VERDICT_LABEL}**\n')
lines.append('Supporting counts:\n')
lines.append(f'- File-backed RNA ALT-positive samples: {assessment["n_positive"]}')
lines.append(f'- File-backed RNA ALT-negative samples: {assessment["n_negative"]}')
lines.append(f'- Cohorts contributing to ALT-positive arm (file-backed): '
             f"{assessment['cohorts_per_arm'][ALT_POSITIVE_LABEL]}")
lines.append(f'- Cohorts contributing to ALT-negative arm (file-backed): '
             f"{assessment['cohorts_per_arm'][ALT_NEGATIVE_LABEL]}")
lines.append(f'- Cohorts carrying both arms (same data_type): '
             f"{assessment['cohorts_with_both_arms_same_dtype']}")
lines.append('')
lines.append('Justification:\n')
lines.append(f'> {verdict["justification"]}')
lines.append('')

REPORT_PATH.write_text('\n'.join(lines))
print(f'Wrote {REPORT_PATH.relative_to(ROOT)}  ({REPORT_PATH.stat().st_size} bytes)')
print(f'Verdict: {VERDICT_LABEL}')

Wrote data/processed/notebook_07_report.md  (13605 bytes)
Verdict: NOT ESTIMABLE AS DEFINED


## 9 \u2014 Idempotence check

Re-run the report-generation logic in memory and confirm the bytes match what we just wrote.

In [10]:
import hashlib

first_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
REPORT_PATH.write_text('\n'.join(lines))  # second write of same string
second_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()

print(f'first  sha256: {first_hash}')
print(f'second sha256: {second_hash}')
assert first_hash == second_hash, 'Notebook 07 is not idempotent \u2014 investigate.'
print('Idempotent: re-writing produces identical bytes.')

first  sha256: 05d1c5f815bfa21647e707dd13345e8ef4a9e6e693dfede6fc34ace4c322d707
second sha256: 05d1c5f815bfa21647e707dd13345e8ef4a9e6e693dfede6fc34ace4c322d707
Idempotent: re-writing produces identical bytes.
